# PoliMillionaire baseline

Before running this notebook, put the files in Google Drive like this:

```
MyDrive/
└── Colab Notebooks/
    └── NLP_assignment/
        ├── poli_millionaire_clean_baseline_v2.ipynb
        └── millionaire_client/
            ├── __init__.py
            ├── client.py
            ├── auth.py
            ├── base.py
            ├── game.py
            ├── models.py
            ├── competitions.py
            ├── leaderboard.py
            └── exceptions.py
```

In [1]:
from google.colab import drive
drive.mount('/content/gdrive/')

import os
import sys
import time
import re
import torch

Mounted at /content/gdrive/


In [2]:
BASE_DIR = '/content/gdrive/MyDrive/NLP_assignment'
PACKAGE_DIR = os.path.join(BASE_DIR, 'millionaire_client')

print('BASE_DIR exists:', os.path.exists(BASE_DIR))
if os.path.exists(BASE_DIR):
    print('BASE_DIR contents:', os.listdir(BASE_DIR))

print('PACKAGE_DIR exists:', os.path.exists(PACKAGE_DIR))
if os.path.exists(PACKAGE_DIR):
    print('PACKAGE_DIR contents:', os.listdir(PACKAGE_DIR))

if not os.path.exists(BASE_DIR):
    raise FileNotFoundError('BASE_DIR not found. Create the NLP_assignment folder in Drive and upload the notebook there.')

if not os.path.exists(PACKAGE_DIR):
    raise FileNotFoundError('millionaire_client folder not found inside BASE_DIR.')

if BASE_DIR not in sys.path:
    sys.path.append(BASE_DIR)

print('Path added successfully.')

BASE_DIR exists: True
BASE_DIR contents: ['PoliMillionaire.ipynb', '.DS_Store', 'millionaire_client', '.ipynb_checkpoints', 'test3_rag_game_runs', 'test3_offline_rag_index.pkl']
PACKAGE_DIR exists: True
PACKAGE_DIR contents: ['base.py', 'leaderboard.py', 'auth.py', 'competitions.py', 'client.py', 'game.py', '__init__.py', 'exceptions.py', 'models.py', '__pycache__']
Path added successfully.


In [3]:
!pip install -q transformers accelerate bitsandbytes sentencepiece protobuf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.1 MB/s eta 0:00:00:00:0100:01


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from millionaire_client import MillionaireClient
from millionaire_client.exceptions import TimeoutError, RateLimitError

In [5]:
API_URL = 'http://131.175.15.22:51111/'
USERNAME = 'gary'
PASSWORD = '13790229'

client = MillionaireClient(API_URL)
user = client.login(USERNAME, PASSWORD)
print('Logged in as:', user.username)

Logged in as: gary


In [6]:
competitions = client.competitions.list_all()
for c in competitions:
    print(c.id, c.name, c.max_levels)

COMPETITION_ID = competitions[3].id

0 Entertainment 15
1 Ancient History and Politics 15
2 Science and Nature 15
3 Maths 15
4 Philosophy and Psychology 15
5 News 15


In [7]:
model_id = 'deepseek-ai/deepseek-math-7b-rl'

# Reverted strictly back to the optimized 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Added attn_implementation="sdpa" to speed up attention calculation overhead
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto',
    attn_implementation="sdpa" 
)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/626 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.61M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.8k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(102400, 4096)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-06)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-06)
      )
    )
    (norm): LlamaRM

In [17]:
MAX_TOKENS = 500

def extract_letter(text):
    # Try to find a specific final answer indicator, spanning across newlines if necessary
    match = re.search(r'(?:final answer|answer is).*\b([ABCD])\b', text, re.IGNORECASE | re.DOTALL)
    if match:
        return match.group(1).upper()
    
    # Fallback: get all standalone A, B, C, or D occurrences and take the LAST one
    matches = re.findall(r'\b([ABCD])\b', text.upper())
    if matches:
        return matches[-1]
        
    return 'A'

def choose_answer(question):
    if len(question.options) < 4:
        return question.options[0].id, 'A', 'fallback'

    prompt = f'''PoliMillionaire MCQ. Reason through the problem step by step (do not write too long due to token limits which is {MAX_TOKENS} tokens), and then provide your final answer as a single letter (A, B, C, or D).

Question: {question.text}
A) {question.options[0].text}
B) {question.options[1].text}
C) {question.options[2].text}
D) {question.options[3].text}

Answer:'''

    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_TOKENS, # Keeping the faster 400 token cap
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    text = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    letter = extract_letter(text)
    idx = ['A', 'B', 'C', 'D'].index(letter)
    return question.options[idx].id, letter, text

In [18]:
def play_game():
    game = client.game.start(competition_id=COMPETITION_ID, mode='text')

    while game.in_progress:
        question = game.current_question
        if question is None:
            break

        print('Level:', game.current_level)
        print(question.text)
        for i, opt in enumerate(question.options):
            print(f"{chr(65+i)}) {opt.text}")

        # extract_letter reverted, so we pass just (question) to choose_answer
        option_id, letter, raw = choose_answer(question)
        print('Predicted:', letter, '| Raw output:', raw)

        max_retries = 5
        for attempt in range(max_retries):
            try:
                result = game.answer(option_id)
                break
            except RateLimitError:
                wait_time = 10 * (attempt + 1)  # Increase the wait time gradually
                print(f'Rate limited, waiting {wait_time} seconds...')
                time.sleep(wait_time)
            except TimeoutError:
                print('Timed out')
                break
        else:
            print('Max retries exceeded for rate limit.')
            break
            
        if 'result' not in locals() or result is None:
            break

        print('Correct:', result.correct, '| Earned:', result.earned_amount)

        if result.game_over:
            break

        time.sleep(0.5)

    return print('Final earned:', game.earned_amount)

In [20]:
play_game()

Level: 1
Suppose that for some $a,b,c$ we have $a+b+c = 1$, $ab+ac+bc = abc = -4$. What is $a^3+b^3+c^3$?
A) 1
B) 112
C) 12
D) 0
Predicted: A | Raw output:  We can use the identity $a^3+b^3+c^3-3abc = (a+b+c)(a^2+b^2+c^2-ab-ac-bc)$. We know that $a+b+c = 1$ and $ab+ac+bc = -4$, and we are given that $abc = -4$. We need to find $a^3+b^3+c^3$.

First, we need to find $a^2+b^2+c^2$. We know that $(a+b+c)^2 = a^2+b^2+c^2+2(ab+ac+bc)$. Substituting the given values, we get $1^2 = a^2+b^2+c^2+2(-4)$, so $a^2+b^2+c^2 = 1+8 = 9$.

Now we can substitute into the identity: $a^3+b^3+c^3-3abc = (a+b+c)(a^2+b^2+c^2-ab-ac-bc)$. We know that $a+b+c = 1$, $ab+ac+bc = -4$, and $abc = -4$. So we have $a^3+b^3+c^3-3(-4) = 1(9-(-4))$, which simplifies to $a^3+b^3+c^3+12 = 13$, so $a^3+b^3+c^3 = 1$. The answer is $\boxed{A}$.
Correct: True | Earned: 100
Level: 2
Determine whether the polynomial in Z[x] satisfies an Eisenstein criterion for irreducibility over Q. 8x^3 + 6x^2 - 9x + 24
A) Yes, with p=2.
B) N

In [11]:
''' --- IGNORE ---
import gc
import torch

try:
    del model
    del tokenizer
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()
print("CUDA VRAM emptied.")
''' 

' --- IGNORE ---\nimport gc\nimport torch\n\ntry:\n    del model\n    del tokenizer\nexcept NameError:\n    pass\n\ngc.collect()\ntorch.cuda.empty_cache()\nprint("CUDA VRAM emptied.")\n'

In [12]:
import torch
free, total = torch.cuda.mem_get_info()
print("Free GB:", free / 1024**3)
print("Total GB:", total / 1024**3)
print("Allocated GB:", torch.cuda.memory_allocated() / 1024**3)
print("Reserved GB:", torch.cuda.memory_reserved() / 1024**3)

Free GB: 2.24786376953125
Total GB: 14.56317138671875
Allocated GB: 4.489849090576172
Reserved GB: 12.1875
